# 3.20 — K-Nearest Neighbors

K-Nearest Neighbors (kNN) predicts by keeping the training examples and asking which stored points are closest to a new point. There is no fitted equation with learned coefficients; the model is the data, the distance rule, the choice of `k`, and the validation score that tells us whether a local vote or average will generalize.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build k-nearest neighbors one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is small enough to inspect by hand. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, distances, sorting, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any synthetic points.

### 1. kNN stores examples instead of fitting a parametric equation

The training set is a memory: each row is a point, and each label is the answer attached to that point. To classify a new point, kNN does not estimate slopes or weights. It asks, "Which stored examples live near this query?" That makes the hypothesis family concrete: the decision is entirely controlled by the training examples, the distance function, and the neighborhood size.

In [ ]:
X_w = np.array([[0.0, 0.2], [0.4, 0.1], [0.2, 0.7],
                [2.0, 2.2], [2.4, 1.9], [1.7, 2.5]])  # two visible clusters.
y_w = np.array([0, 0, 0, 1, 1, 1])  # class labels attached to stored examples.
q_w = np.array([1.4, 1.6])  # a new point whose label we want to infer.

print("training shape:", X_w.shape)  # six stored examples, two features.
print("query:", q_w)  # the only new object to classify.

▶ What you'll see: six remembered points and one query point; no coefficients are fitted.

In [ ]:
plt.figure(figsize=(4.5, 3.5))
plt.scatter(X_w[y_w == 0, 0], X_w[y_w == 0, 1], s=80, label="class 0", color="steelblue")
plt.scatter(X_w[y_w == 1, 0], X_w[y_w == 1, 1], s=80, label="class 1", color="darkorange")
plt.scatter([q_w[0]], [q_w[1]], s=140, marker="*", color="black", label="query")
plt.title("1: stored examples plus one query")
plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.legend(); plt.show()

▶ What you'll see: the query sits nearer to the orange cluster than to the blue cluster.

*Why it's done this way:* kNN is a nonparametric rule: instead of compressing the training set into a small parameter vector, it delays the work until prediction time. That is powerful when local geometry is meaningful, but it also means distance and scale become part of the model, not implementation details.

### 2. Distance turns "near" into arithmetic

The usual distance is Euclidean distance. For a query $x$ and stored example $x_i$, the formula is

$$d(x,x_i)=\sqrt{\sum_j (x_j-x_{ij})^2}.$$

Subtract coordinate by coordinate, square so opposite signs do not cancel, sum the squared gaps, and take a square root to return to the original feature units.

In [ ]:
diffs_w = X_w - q_w  # coordinate-wise gaps from every stored point to the query.
sq_w = diffs_w ** 2  # squared gaps, one row per stored point.

print("first point diff:", np.round(diffs_w[0], 3))
print("first point squared gaps:", np.round(sq_w[0], 3))

▶ What you'll see: the first point is far from the query in both coordinates, and squaring makes both gaps positive.

In [ ]:
dists_w = np.sqrt(np.sum(sq_w, axis=1))  # Euclidean distance to every stored example.

print("distances:", np.round(dists_w, 3))

assert np.allclose(np.round(dists_w, 3), [1.980, 1.803, 1.5, 0.849, 1.044, 0.949])

▶ What you'll see: the smallest distances are to class-1 examples, especially the point `[2.0, 2.2]`.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(np.arange(len(dists_w)), dists_w, color=np.where(y_w == 1, "darkorange", "steelblue"))
plt.title("2: distance from query to each stored point")
plt.xlabel("training index"); plt.ylabel("Euclidean distance"); plt.show()

▶ What you'll see: low orange bars are the nearby evidence that will dominate the vote.

*Why it's done this way:* the square-and-sum formula is the Pythagorean theorem in feature space. It rewards points that are simultaneously close on all relevant coordinates; one very large coordinate gap can dominate, which is why feature scaling is not optional for kNN.

### 3. The nearest-neighbor set $N_k(x)$

The formula in the lesson is $\hat y(x)=\operatorname{mode}\{y_i:i\in N_k(x)\}$. The set $N_k(x)$ is built by sorting distances and keeping the first `k` indices. Classification then uses the most common label among those neighbors.

In [ ]:
order_w = np.argsort(dists_w)  # indices from nearest to farthest.
k_w = 3  # neighborhood size.
neighbors_w = order_w[:k_w]  # N_k(x), the three closest stored examples.

print("sorted indices:", order_w)
print("3 nearest indices:", neighbors_w)

▶ What you'll see: the first three sorted indices are the local evidence for the query.

In [ ]:
neighbor_labels_w = y_w[neighbors_w]  # labels inside N_k(x).
counts_w = np.bincount(neighbor_labels_w, minlength=2)  # class counts for the mode.
pred_w = int(np.argmax(counts_w))  # majority vote.

print("neighbor labels:", neighbor_labels_w)
print("class counts:", counts_w)
print("predicted class:", pred_w)

assert pred_w == 1

▶ What you'll see: all three nearest neighbors are class 1, so the mode is class 1.

In [ ]:
plt.figure(figsize=(4.5, 3.5))
plt.scatter(X_w[:, 0], X_w[:, 1], c=y_w, cmap="coolwarm", s=80)
plt.scatter([q_w[0]], [q_w[1]], s=150, marker="*", color="black")
for idx_w in neighbors_w:
    plt.plot([q_w[0], X_w[idx_w, 0]], [q_w[1], X_w[idx_w, 1]], color="gray", linewidth=1.5)
plt.title("3: the three nearest neighbors vote")
plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.show()

▶ What you'll see: three gray line segments connect the query to the neighbors that decide the label.

*Why it's done this way:* sorting distances converts geometry into a discrete evidence set. The mode is the empirical-risk choice for 0/1 classification inside that local set: if every neighbor is treated as one local sample, the label with the most votes makes the fewest mistakes on that neighborhood.

### 4. k controls flexibility and stability

A small `k` follows the closest point and can change abruptly; a larger `k` smooths over local noise but can ignore real small regions. This is the recurring Part 3 bargain: flexibility helps fit the current sample, while stability helps future examples.

In [ ]:
ks_w = np.array([1, 3, 5])  # compare increasingly broad neighborhoods.
for kk_w in ks_w:
    nn_w = order_w[:kk_w]
    counts_kk_w = np.bincount(y_w[nn_w], minlength=2)
    print("k=", kk_w, "neighbors=", nn_w, "counts=", counts_kk_w, "pred=", int(np.argmax(counts_kk_w)))

▶ What you'll see: the prediction is stable here because the closest region is strongly class 1.

In [ ]:
noisy_X_w = np.vstack([X_w, [1.35, 1.55]])  # add one suspicious class-0 point very near the query.
noisy_y_w = np.append(y_w, 0)
noisy_d_w = np.sqrt(np.sum((noisy_X_w - q_w) ** 2, axis=1))
noisy_order_w = np.argsort(noisy_d_w)
for kk_w in ks_w:
    nn_w = noisy_order_w[:kk_w]
    counts_kk_w = np.bincount(noisy_y_w[nn_w], minlength=2)
    print("with noisy point, k=", kk_w, "labels=", noisy_y_w[nn_w], "pred=", int(np.argmax(counts_kk_w)))

▶ What you'll see: `k=1` follows the suspicious point, while larger `k` can resist it.

In [ ]:
plt.figure(figsize=(4.5, 3.5))
plt.scatter(noisy_X_w[noisy_y_w == 0, 0], noisy_X_w[noisy_y_w == 0, 1], s=80, color="steelblue", label="class 0")
plt.scatter(noisy_X_w[noisy_y_w == 1, 0], noisy_X_w[noisy_y_w == 1, 1], s=80, color="darkorange", label="class 1")
plt.scatter([q_w[0]], [q_w[1]], marker="*", s=150, color="black", label="query")
plt.title("4: one near noisy point can flip k=1")
plt.legend(); plt.show()

▶ What you'll see: the blue point next to the query is exactly the kind of local noise that small `k` trusts too much.

*Why it's done this way:* `k` is a capacity knob. `k=1` has low bias but high variance because one sample can decide everything; larger `k` averages more evidence, increasing bias but reducing variance.

### 5. Scaling decides which feature gets to matter

Distances are computed in raw feature units. If one feature is measured in thousands and another in decimals, the large-scale feature can dominate the square-and-sum formula even if it is not more informative.

In [ ]:
X_scale_w = np.array([[0.0, 100.0], [1.0, 120.0], [2.0, 900.0], [3.0, 920.0]])
y_scale_w = np.array([0, 0, 1, 1])
q_scale_w = np.array([2.2, 130.0])
raw_d_w = np.sqrt(np.sum((X_scale_w - q_scale_w) ** 2, axis=1))

print("raw distances:", np.round(raw_d_w, 2))
print("raw nearest label:", y_scale_w[np.argmin(raw_d_w)])

▶ What you'll see: the second feature's hundreds-of-units scale drives the nearest neighbor.

In [ ]:
mean_w = X_scale_w.mean(axis=0)
std_w = X_scale_w.std(axis=0)
Z_scale_w = (X_scale_w - mean_w) / std_w
zq_w = (q_scale_w - mean_w) / std_w
scaled_d_w = np.sqrt(np.sum((Z_scale_w - zq_w) ** 2, axis=1))

print("scaled distances:", np.round(scaled_d_w, 3))
print("scaled nearest label:", y_scale_w[np.argmin(scaled_d_w)])

▶ What you'll see: after standardization, both features get comparable influence.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(np.arange(4) - 0.18, raw_d_w / raw_d_w.max(), width=0.36, label="raw normalized")
plt.bar(np.arange(4) + 0.18, scaled_d_w / scaled_d_w.max(), width=0.36, label="standardized")
plt.title("5: scaling changes distances")
plt.xlabel("training index"); plt.ylabel("relative distance"); plt.legend(); plt.show()

▶ What you'll see: the ranking of neighbors can change once units stop dominating geometry.

*Why it's done this way:* standardization subtracts a mean and divides by a standard deviation, turning each coordinate into "how many typical spreads away." That makes the Euclidean sum compare dimensionless deviations rather than incompatible units.

### 6. Validation scores choose the usable rule

The content block gives a tiny empirical-risk calculation: losses `0.202`, `0.096`, and `0.437` average to `0.245`; adding a method cost `0.070` gives `0.315`. We reproduce that arithmetic and compare it with a more flexible alternative and a stabilized setting.

In [ ]:
losses_score_w = np.array([0.202, 0.096, 0.437])  # three verified per-example losses.
R_S_w = float(np.mean(losses_score_w))  # empirical risk: average loss.
cost_w = 0.070  # complexity, regularization, or operational cost.
score_w = R_S_w + cost_w

print("empirical risk:", round(R_S_w, 3))
print("decision score:", round(score_w, 3))

assert round(R_S_w, 3) == 0.245 and round(score_w, 3) == 0.315

▶ What you'll see: the raw training average is not the final decision score; cost changes the number to compare.

In [ ]:
flex_w = 0.351
gap_w = flex_w - score_w
relative_gap_w = gap_w / flex_w
stable_w = 0.80 * score_w

print("gap:", round(gap_w, 3), "relative gap:", round(relative_gap_w, 3))
print("stabilized score:", round(stable_w, 3))

assert round(gap_w, 3) == 0.036 and round(relative_gap_w, 3) == 0.103 and round(stable_w, 3) == 0.252

▶ What you'll see: the stabilized score is lowest in this toy comparison.

In [ ]:
names_w = ["baseline", "flexible", "stabilized"]
scores_w = np.array([score_w, flex_w, stable_w])
plt.figure(figsize=(4.8, 3))
plt.bar(names_w, scores_w, color=["gray", "indianred", "seagreen"])
plt.ylabel("selection score (lower is better)")
plt.title("6: validation-style score chooses the rule")
plt.show()

print("winner:", names_w[int(np.argmin(scores_w))])

▶ What you'll see: the green stabilized bar is the smallest, so that setting would be carried forward.

*Why it's done this way:* kNN can always chase local training quirks by shrinking `k` or changing distance. A validation-style score forces the comparison to include loss plus the cost of flexibility, so the selected neighborhood rule is judged by expected future behavior rather than by the prettiest training fragment.


## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses a
> handful of small numbers, prints every intermediate value with inline `# ->` checks, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Store labeled examples

kNN's "fit" step is just memory: keep the feature rows, keep their labels, and wait for a query.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_X = np.array([[0.0, 0.0], [0.5, 0.2], [1.0, 0.1],
                 [3.0, 3.0], [3.5, 3.2], [4.0, 2.8]])  # -> six stored points
t1_y = np.array([0, 0, 0, 1, 1, 1])  # -> three labels per class
t1_query = np.array([2.7, 2.6])  # -> one new point
t1_counts = np.bincount(t1_y, minlength=2)  # -> [3, 3]

print("stored shape:", t1_X.shape)  # -> (6, 2)
print("labels:", t1_y.tolist())  # -> [0, 0, 0, 1, 1, 1]
print("class counts:", t1_counts.tolist())  # -> [3, 3]
print("query:", t1_query.tolist())  # -> [2.7, 2.6]

assert t1_X.shape == (6, 2)
assert t1_counts.tolist() == [3, 3]

plt.figure(figsize=(4.6, 3.4))
plt.scatter(t1_X[t1_y == 0, 0], t1_X[t1_y == 0, 1], color="steelblue", label="class 0")
plt.scatter(t1_X[t1_y == 1, 0], t1_X[t1_y == 1, 1], color="darkorange", label="class 1")
plt.scatter([t1_query[0]], [t1_query[1]], marker="*", s=160, color="black", label="query")
plt.title("Toy 1 · kNN memory plus query")
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.legend()
plt.show()

▶ What you'll see: six remembered training points and one black query star; no parameters are fitted.

### ✍️ Toy 2 · Compute Euclidean distances

Distance turns "near" into arithmetic. Subtract the query, square coordinate gaps, sum them, and
take square roots.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_X = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [3.0, 2.0], [4.0, 2.0], [3.0, 3.0]])  # -> six stored points
t2_query = np.array([2.0, 1.5])  # -> query point
t2_diffs = t2_X - t2_query  # -> row-wise coordinate gaps
t2_squared = t2_diffs ** 2  # -> squared gaps
t2_sq_sums = t2_squared.sum(axis=1)  # -> [6.25, 3.25, 4.25, 1.25, 4.25, 3.25]
t2_distances = np.sqrt(t2_sq_sums)  # -> [2.5, 1.803, 2.062, 1.118, 2.062, 1.803]

print("diffs:\n", t2_diffs)  # -> [[-2.0, -1.5], [-1.0, -1.5], ...]
print("squared gaps:\n", t2_squared)  # -> nonnegative coordinate gaps
print("squared sums:", np.round(t2_sq_sums, 3).tolist())  # -> [6.25, 3.25, 4.25, 1.25, 4.25, 3.25]
print("distances:", np.round(t2_distances, 3).tolist())  # -> [2.5, 1.803, 2.062, 1.118, 2.062, 1.803]

assert np.allclose(np.round(t2_distances, 3), [2.5, 1.803, 2.062, 1.118, 2.062, 1.803])

plt.figure(figsize=(4.7, 3.0))
plt.bar(np.arange(t2_distances.size), t2_distances, color="slateblue")
plt.title("Toy 2 · distances to the query")
plt.xlabel("training index")
plt.ylabel("Euclidean distance")
plt.show()

▶ What you'll see: training index 3 has the shortest distance bar at about `1.118`.

### ✍️ Toy 3 · Sort neighbors and vote

After distances are known, `argsort` builds the neighbor set. Classification is the majority label
inside that set.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_X = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [3.0, 2.0], [4.0, 2.0], [3.0, 3.0]])  # -> six stored points
t3_y = np.array([0, 0, 0, 1, 1, 1])  # -> stored labels
t3_query = np.array([2.0, 1.5])  # -> query point
t3_distances = np.sqrt(np.sum((t3_X - t3_query) ** 2, axis=1))  # -> [2.5, 1.803, 2.062, 1.118, 2.062, 1.803]
t3_order = np.argsort(t3_distances)  # -> [3, 1, 5, 2, 4, 0]
t3_k = 3  # -> three neighbors
t3_neighbors = t3_order[:t3_k]  # -> [3, 1, 5]
t3_neighbor_labels = t3_y[t3_neighbors]  # -> [1, 0, 1]
t3_counts = np.bincount(t3_neighbor_labels, minlength=2)  # -> [1, 2]
t3_pred = int(np.argmax(t3_counts))  # -> 1

print("distances:", np.round(t3_distances, 3).tolist())  # -> [2.5, 1.803, 2.062, 1.118, 2.062, 1.803]
print("sorted indices:", t3_order.tolist())  # -> [3, 1, 5, 2, 4, 0]
print("neighbors:", t3_neighbors.tolist())  # -> [3, 1, 5]
print("neighbor labels:", t3_neighbor_labels.tolist())  # -> [1, 0, 1]
print("vote counts:", t3_counts.tolist())  # -> [1, 2]
print("prediction:", t3_pred)  # -> 1

assert t3_pred == 1

plt.figure(figsize=(4.6, 3.4))
plt.scatter(t3_X[:, 0], t3_X[:, 1], c=t3_y, cmap="coolwarm", s=80)
plt.scatter([t3_query[0]], [t3_query[1]], marker="*", s=160, color="black")
for t3_idx in t3_neighbors:
    plt.plot([t3_query[0], t3_X[t3_idx, 0]], [t3_query[1], t3_X[t3_idx, 1]], color="gray")
plt.title("Toy 3 · three neighbors vote")
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.show()

▶ What you'll see: two of the three connected neighbors are class 1, so the local vote predicts class 1.

### ✍️ Toy 4 · Change k to resist one noisy point

A very close mislabeled point can dominate `k=1`. Larger neighborhoods can outvote that local noise.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_X = np.array([[0.0, 0.0], [0.2, 0.1], [0.5, -0.1],
                 [1.0, 1.2], [1.3, 1.1], [1.4, 1.3], [1.19, 1.14]])  # -> six regular points plus one near noise
t4_y = np.array([0, 0, 0, 1, 1, 1, 0])  # -> the nearest point is class 0 noise
t4_query = np.array([1.2, 1.15])  # -> query beside the noisy point
t4_distances = np.sqrt(np.sum((t4_X - t4_query) ** 2, axis=1))  # -> distances to all seven points
t4_order = np.argsort(t4_distances)  # -> [6, 4, 3, 5, 2, 1, 0]
t4_ks = np.array([1, 3, 5])  # -> compare three neighborhoods
t4_preds = []

print("sorted labels:", t4_y[t4_order].tolist())  # -> [0, 1, 1, 1, 0, 0, 0]

for t4_k in t4_ks:
    t4_neighbors = t4_order[:t4_k]
    t4_labels = t4_y[t4_neighbors]
    t4_counts = np.bincount(t4_labels, minlength=2)
    t4_pred = int(np.argmax(t4_counts))
    t4_preds.append(t4_pred)
    print("k=", int(t4_k), "neighbors", t4_neighbors.tolist(), "labels", t4_labels.tolist(), "counts", t4_counts.tolist(), "pred", t4_pred)
t4_preds = np.array(t4_preds)  # -> [0, 1, 1]
assert t4_preds.tolist() == [0, 1, 1]

plt.figure(figsize=(4.6, 3.4))
plt.scatter(t4_X[t4_y == 0, 0], t4_X[t4_y == 0, 1], color="steelblue", label="class 0")
plt.scatter(t4_X[t4_y == 1, 0], t4_X[t4_y == 1, 1], color="darkorange", label="class 1")
plt.scatter([t4_query[0]], [t4_query[1]], marker="*", s=160, color="black", label="query")
plt.scatter([t4_X[6, 0]], [t4_X[6, 1]], s=140, facecolors="none", edgecolors="crimson", linewidths=2, label="near noise")
plt.title("Toy 4 · larger k outvotes noise")
plt.xlabel("feature 0")
plt.ylabel("feature 1")
plt.legend()
plt.show()

▶ What you'll see: `k=1` follows the near blue noisy point, while `k=3` and `k=5` predict class 1.

### ✍️ Toy 5 · Standardize before measuring distance

Raw units can dominate distances. After standardization, a feature measured on a huge scale no
longer drowns out the other coordinate.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_X = np.array([[0.0, 100.0], [1.0, 120.0], [2.0, 500.0],
                 [3.0, 520.0], [4.0, 900.0], [5.0, 920.0]])  # -> second feature has huge units
t5_y = np.array([0, 0, 1, 1, 0, 0])  # -> labels
t5_query = np.array([3.1, 130.0])  # -> raw second feature near index 1
t5_raw_distances = np.sqrt(np.sum((t5_X - t5_query) ** 2, axis=1))  # -> raw Euclidean distances
t5_mean = t5_X.mean(axis=0)  # -> [2.5, 510.0]
t5_std = t5_X.std(axis=0)  # -> [1.708, 326.752]
t5_Z = (t5_X - t5_mean) / t5_std  # -> standardized training rows
t5_z_query = (t5_query - t5_mean) / t5_std  # -> [0.351, -1.163]
t5_scaled_distances = np.sqrt(np.sum((t5_Z - t5_z_query) ** 2, axis=1))  # -> standardized distances
t5_raw_nearest = int(np.argmin(t5_raw_distances))  # -> 1
t5_scaled_nearest = int(np.argmin(t5_scaled_distances))  # -> 3

print("raw distances:", np.round(t5_raw_distances, 1).tolist())  # -> [30.2, 10.2, 370.0, 390.0, 770.0, 790.0]
print("mean:", np.round(t5_mean, 3).tolist())  # -> [2.5, 510.0]
print("std:", np.round(t5_std, 3).tolist())  # -> [1.708, 326.752]
print("standardized query:", np.round(t5_z_query, 3).tolist())  # -> [0.351, -1.163]
print("scaled distances:", np.round(t5_scaled_distances, 3).tolist())  # -> [1.817, 1.23, 1.303, 1.195, 2.415, 2.661]
print("raw nearest label:", int(t5_y[t5_raw_nearest]))  # -> 0
print("scaled nearest label:", int(t5_y[t5_scaled_nearest]))  # -> 1

assert t5_raw_nearest == 1 and t5_scaled_nearest == 3

plt.figure(figsize=(4.8, 3.0))
plt.bar(np.arange(t5_X.shape[0]) - 0.18, t5_raw_distances / t5_raw_distances.max(), width=0.36, label="raw")
plt.bar(np.arange(t5_X.shape[0]) + 0.18, t5_scaled_distances / t5_scaled_distances.max(), width=0.36, label="standardized")
plt.title("Toy 5 · scaling changes neighbor rank")
plt.xlabel("training index")
plt.ylabel("relative distance")
plt.legend()
plt.show()

▶ What you'll see: raw distance chooses label 0 at index 1, while standardized distance chooses label 1 at index 3.

### ✍️ Toy 6 · Validate k with loss plus cost

A validation-style score compares candidate `k` values by prediction loss plus a simple method cost.
Here larger `k` wins because it is accurate and less flexible.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_X_train = np.array([[0.0, 0.0], [0.3, 0.2], [0.6, -0.1], [1.0, 0.2],
                       [2.5, 2.4], [2.8, 2.6], [3.1, 2.7], [3.4, 3.0]])  # -> eight training rows
t6_y_train = np.array([0, 0, 0, 1, 1, 1, 1, 1])  # -> one local noisy label near class 0
t6_X_val = np.array([[0.9, 0.1], [2.7, 2.5], [0.2, 0.1], [3.2, 2.8]])  # -> four validation points
t6_y_val = np.array([0, 1, 0, 1])  # -> validation labels
t6_ks = np.array([1, 3, 5])  # -> candidate neighborhoods
t6_costs = np.array([0.09, 0.04, 0.02])  # -> smaller k gets a flexibility cost
t6_scores = []
t6_risks = []
for t6_k, t6_cost in zip(t6_ks, t6_costs):
    t6_preds = []
    for t6_query in t6_X_val:
        t6_distances = np.sqrt(np.sum((t6_X_train - t6_query) ** 2, axis=1))
        t6_neighbors = np.argsort(t6_distances)[:t6_k]
        t6_counts = np.bincount(t6_y_train[t6_neighbors], minlength=2)
        t6_preds.append(int(np.argmax(t6_counts)))
    t6_preds = np.array(t6_preds)
    t6_losses = (t6_preds != t6_y_val).astype(int)
    t6_risk = float(t6_losses.mean())
    t6_score = t6_risk + t6_cost
    t6_risks.append(t6_risk)
    t6_scores.append(t6_score)
    print("k=", int(t6_k), "preds", t6_preds.tolist(), "losses", t6_losses.tolist(), "risk", round(t6_risk, 3), "cost", round(float(t6_cost), 3), "score", round(t6_score, 3))
t6_risks = np.array(t6_risks)  # -> [0.25, 0.0, 0.0]
t6_scores = np.array(t6_scores)  # -> [0.34, 0.04, 0.02]
t6_best = int(np.argmin(t6_scores))  # -> 2

print("risks:", np.round(t6_risks, 3).tolist())  # -> [0.25, 0.0, 0.0]
print("scores:", np.round(t6_scores, 3).tolist())  # -> [0.34, 0.04, 0.02]
print("chosen k:", int(t6_ks[t6_best]))  # -> 5

assert int(t6_ks[t6_best]) == 5

plt.figure(figsize=(4.6, 3.0))
plt.bar(["k=1", "k=3", "k=5"], t6_scores, color=["crimson", "gray", "seagreen"])
plt.title("Toy 6 · validation score selects k")
plt.ylabel("loss + cost")
plt.show()

▶ What you'll see: `k=5` has the lowest complete score after validation loss and flexibility cost are added.

## 🛠️ Setup

In [ ]:
import numpy as np # Import NumPy for arrays, distance calculations, sorting, voting, and numerical checks.
import matplotlib.pyplot as plt # Import Matplotlib for the scatter plots, bar charts, and decision-region visualizations.
np.random.seed(0) # Fix the global random seed so every stochastic example is repeatable.

## 🟢 Basics (warm-up)

### Basic 1 — Store a labeled training set

**Goal.** Put examples and labels into arrays, because kNN predicts from stored rows rather than from fitted coefficients. We build it in 2 steps.

In [ ]:
X_b1 = np.array([[0.0, 0.2], [0.4, 0.1], [0.2, 0.7], [2.0, 2.2], [2.4, 1.9], [1.7, 2.5]]) # Store six 2-D training examples.
y_b1 = np.array([0, 0, 0, 1, 1, 1]) # Attach one class label to each stored row.

print("X shape:", X_b1.shape) # Inspect the number of rows and features.
print("labels:", y_b1) # Inspect the remembered answers.

▶ What you'll see: six examples with two features and binary labels.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact scatter plot.
plt.scatter(X_b1[:, 0], X_b1[:, 1], c=y_b1, cmap="coolwarm", s=80) # Color each point by its stored label.
plt.title("Basic 1: labeled memory") # Title the example plot.
plt.xlabel("feature 0") # Label the first feature.
plt.ylabel("feature 1") # Label the second feature.
plt.show() # Display the stored examples.

▶ What you'll see: two small clusters that kNN can use as local evidence.

👀 Takeaway: kNN's model is the labeled training set plus a distance rule.

### Basic 2 — Compute one Euclidean distance

**Goal.** Compute the distance between one query and one stored point, because every kNN prediction starts with this arithmetic. We build it in 2 steps.

In [ ]:
x_b2 = np.array([0.4, 0.1]) # Choose one stored point.
q_b2 = np.array([1.4, 1.6]) # Choose the query point.
diff_b2 = q_b2 - x_b2 # Subtract coordinates to measure gaps.

print("coordinate gaps:", diff_b2) # Inspect differences before squaring.

▶ What you'll see: the query is 1.0 units away on feature 0 and 1.5 units away on feature 1.

In [ ]:
sq_b2 = diff_b2 ** 2 # Square each coordinate gap.
dist_b2 = float(np.sqrt(np.sum(sq_b2))) # Sum squared gaps and take a square root.

print("squared gaps:", sq_b2) # Inspect the pieces inside the distance formula.
print("distance:", round(dist_b2, 3)) # Inspect the final Euclidean distance.

assert round(dist_b2, 3) == 1.803 # Verify sqrt(1^2 + 1.5^2).

▶ What you'll see: the distance is about 1.803.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact geometry plot for the distance calculation.
plt.scatter([x_b2[0]], [x_b2[1]], s=90, color="steelblue", label="stored point") # Draw the stored example.
plt.scatter([q_b2[0]], [q_b2[1]], marker="*", s=150, color="black", label="query") # Draw the query.
plt.plot([x_b2[0], q_b2[0]], [x_b2[1], q_b2[1]], color="gray", linewidth=2) # Connect the two points whose distance was computed.
plt.text((x_b2[0] + q_b2[0]) / 2, (x_b2[1] + q_b2[1]) / 2, f"d={dist_b2:.3f}", ha="center", va="bottom") # Label the measured distance.
plt.title("Basic 2: one Euclidean distance") # Title the distance sketch.
plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.axis("equal"); plt.legend(); plt.show() # Display the geometry.

▶ What you'll see: the gray segment is the exact point-to-query distance computed above.

👀 Takeaway: Euclidean distance is a square-rooted sum of squared coordinate gaps.

### Basic 3 — Compute distances to all points

**Goal.** Vectorize distance computation, because kNN needs one distance from the query to every stored example. We build it in 2 steps.

In [ ]:
X_b3 = np.array([[0.0, 0.2], [0.4, 0.1], [0.2, 0.7], [2.0, 2.2], [2.4, 1.9], [1.7, 2.5]]) # Recreate the toy training set.
q_b3 = np.array([1.4, 1.6]) # Define the query.
diffs_b3 = X_b3 - q_b3 # Broadcast the query subtraction across all rows.

print("first two diffs:\n", np.round(diffs_b3[:2], 3)) # Inspect the row-wise gaps.

▶ What you'll see: each row now contains a point-to-query difference vector.

In [ ]:
dists_b3 = np.sqrt(np.sum(diffs_b3 ** 2, axis=1)) # Convert every row of gaps into one distance.

print("all distances:", np.round(dists_b3, 3)) # Inspect distances to all stored points.

plt.figure(figsize=(4, 3)) # Create a compact distance chart.
plt.bar(np.arange(len(dists_b3)), dists_b3, color="slateblue") # Plot one bar per stored example.
plt.title("Basic 3: query-to-training distances") # Title the chart.
plt.xlabel("training index") # Label stored-example indices.
plt.ylabel("distance") # Label distance scale.
plt.show() # Display the distances.

▶ What you'll see: the closest stored examples have the lowest bars.

👀 Takeaway: distance vectorization turns a query into a sortable list of neighbor evidence.

### Basic 4 — Sort nearest neighbors

**Goal.** Use `argsort` to build $N_k(x)$, because kNN needs indices of the closest rows, not just distance values. We build it in 2 steps.

In [ ]:
dists_b4 = np.array([1.980, 1.803, 1.500, 0.849, 1.044, 0.949]) # Reuse rounded distances from the toy geometry.
order_b4 = np.argsort(dists_b4) # Sort indices from nearest to farthest.

print("order:", order_b4) # Inspect the ranking of examples.
print("sorted distances:", dists_b4[order_b4]) # Confirm the distances are ascending.

▶ What you'll see: indices 3, 5, and 4 are nearest.

In [ ]:
k_b4 = 3 # Choose three nearest neighbors.
neighbors_b4 = order_b4[:k_b4] # Slice the first k sorted indices.

print("N_k indices:", neighbors_b4) # Inspect the actual neighbor set.

assert np.array_equal(neighbors_b4, np.array([3, 5, 4])) # Verify the three nearest indices.

▶ What you'll see: the neighbor set is exactly the first three positions in sorted order.

In [ ]:
rank_b4 = np.arange(len(order_b4)) # Positions in the nearest-to-farthest ranking.
colors_b4 = np.where(rank_b4 < k_b4, "darkorange", "lightgray") # Highlight the selected neighborhood.
plt.figure(figsize=(4.5, 3)) # Create a compact sorted-distance chart.
plt.bar(rank_b4, dists_b4[order_b4], color=colors_b4) # Plot distances in neighbor order.
plt.xticks(rank_b4, order_b4) # Label bars by original training index.
plt.title("Basic 4: sorted distances define N_k") # Title the ranking plot.
plt.xlabel("training index in sorted order"); plt.ylabel("distance"); plt.show() # Display the selected neighbors.

▶ What you'll see: the first three orange bars are the neighbors kept in $N_k(x)$.

👀 Takeaway: `argsort` is the bridge from distances to a neighborhood set.

### Basic 5 — Majority vote for classification

**Goal.** Convert neighbor labels into a predicted class, because kNN classification uses the mode inside $N_k(x)$. We build it in 2 steps.

In [ ]:
y_b5 = np.array([0, 0, 0, 1, 1, 1]) # Store labels for six training examples.
neighbors_b5 = np.array([3, 5, 4]) # Use the three nearest neighbors.
labels_b5 = y_b5[neighbors_b5] # Gather labels inside the neighborhood.

print("neighbor labels:", labels_b5) # Inspect the local evidence.

▶ What you'll see: all three nearest neighbors vote for class 1.

In [ ]:
counts_b5 = np.bincount(labels_b5, minlength=2) # Count votes for each class.
pred_b5 = int(np.argmax(counts_b5)) # Choose the class with the largest count.

print("class counts:", counts_b5) # Inspect the vote totals.
print("prediction:", pred_b5) # Inspect the final class.

assert pred_b5 == 1 # Verify the majority vote.

▶ What you'll see: class 1 wins 3 to 0.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact vote-count chart.
plt.bar(["class 0", "class 1"], counts_b5, color=["steelblue", "darkorange"]) # Show the local majority vote.
plt.title("Basic 5: majority vote inside N_k") # Title the vote plot.
plt.ylabel("neighbor votes"); plt.ylim(0, max(counts_b5) + 1); plt.show() # Display the class counts.

▶ What you'll see: class 1 has all the neighbor votes, matching the predicted label.

👀 Takeaway: kNN classification is a local empirical majority vote.

### Basic 6 — Average neighbors for regression

**Goal.** Predict a number instead of a class, because kNN regression uses nearby target values and averages them. We build it in 2 steps.

In [ ]:
X_b6 = np.array([[0.0], [1.0], [2.0], [3.0], [4.0]]) # Store one-dimensional inputs.
y_b6 = np.array([1.0, 1.4, 1.9, 3.2, 3.9]) # Store numeric targets.
q_b6 = np.array([2.6]) # Choose a query location.
dists_b6 = np.abs(X_b6[:, 0] - q_b6[0]) # In 1-D, Euclidean distance is absolute difference.

print("distances:", np.round(dists_b6, 2)) # Inspect closeness to the query.

▶ What you'll see: the points at 2 and 3 are closest to 2.6.

In [ ]:
neighbors_b6 = np.argsort(dists_b6)[:3] # Keep the three nearest target values.
pred_b6 = float(np.mean(y_b6[neighbors_b6])) # Average their numeric targets.

print("neighbors:", neighbors_b6, "targets:", y_b6[neighbors_b6]) # Inspect the local values being averaged.
print("regression prediction:", round(pred_b6, 3)) # Inspect the local mean.

assert round(pred_b6, 3) == 3.0 # Verify (3.2 + 1.9 + 3.9)/3.

▶ What you'll see: the prediction is the mean of the three nearest target values.

In [ ]:
plt.figure(figsize=(4.8, 3)) # Create a compact 1-D regression-neighborhood plot.
plt.scatter(X_b6[:, 0], y_b6, s=70, color="steelblue", label="training targets") # Draw stored numeric targets.
plt.scatter(X_b6[neighbors_b6, 0], y_b6[neighbors_b6], s=130, facecolors="none", edgecolors="darkorange", linewidths=2, label="3 nearest") # Highlight averaged neighbors.
plt.axvline(q_b6[0], color="black", linestyle="--", label="query") # Mark the query location.
plt.axhline(pred_b6, color="seagreen", linestyle=":", label=f"prediction={pred_b6:.1f}") # Mark the neighbor average.
plt.title("Basic 6: local average for regression") # Title the regression plot.
plt.xlabel("x"); plt.ylabel("target"); plt.legend(); plt.show() # Display the local averaging picture.

▶ What you'll see: the highlighted neighbors around the query average to the green prediction line.

👀 Takeaway: kNN regression replaces the mode with a local average.

### Basic 7 — Break ties deliberately

**Goal.** See what happens when votes tie, because even `k` can create ambiguous classification neighborhoods. We build it in 2 steps.

In [ ]:
labels_b7 = np.array([0, 1, 0, 1]) # Create an even-sized neighborhood with two votes per class.
counts_b7 = np.bincount(labels_b7, minlength=2) # Count class votes.

print("counts:", counts_b7) # Inspect the tie.

▶ What you'll see: both classes receive two votes.

In [ ]:
pred_default_b7 = int(np.argmax(counts_b7)) # NumPy returns the first maximum, so lower label wins ties.
odds_b7 = np.array([1, 3, 5]) # Odd k values reduce tie frequency in binary classification.

print("default tie prediction:", pred_default_b7) # Inspect deterministic tie behavior.
print("odd k choices:", odds_b7) # Inspect safer binary k choices.

assert pred_default_b7 == 0 # Verify first-maximum tie break.

▶ What you'll see: the deterministic tie-break picks class 0, even though evidence is balanced.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact tie-vote chart.
plt.bar(["class 0", "class 1"], counts_b7, color=["steelblue", "darkorange"]) # Show balanced votes.
plt.axhline(counts_b7.max(), color="gray", linestyle="--", linewidth=1) # Mark the tied maximum.
plt.title("Basic 7: tied neighborhood vote") # Title the tie plot.
plt.ylabel("neighbor votes"); plt.ylim(0, counts_b7.max() + 1); plt.show() # Display the tie.

▶ What you'll see: both bars reach the same height, so the tie rule rather than evidence decides.

👀 Takeaway: choose tie rules intentionally, and prefer odd `k` for two-class problems when possible.

### Basic 8 — Standardize feature scales

**Goal.** Put features on comparable units, because Euclidean distance can be dominated by the largest-scale coordinate. We build it in 2 steps.

In [ ]:
X_b8 = np.array([[0.0, 100.0], [1.0, 120.0], [2.0, 900.0], [3.0, 920.0]]) # Two features with very different scales.
mean_b8 = X_b8.mean(axis=0) # Compute training means for each feature.
std_b8 = X_b8.std(axis=0) # Compute training standard deviations for each feature.

print("means:", np.round(mean_b8, 2), "stds:", np.round(std_b8, 2)) # Inspect scaling constants.

▶ What you'll see: the second feature has a much larger spread.

In [ ]:
Z_b8 = (X_b8 - mean_b8) / std_b8 # Standardize training rows.

print("standardized first row:", np.round(Z_b8[0], 3)) # Inspect a dimensionless transformed row.

plt.figure(figsize=(4, 3)) # Create a compact before/after scale chart.
plt.bar(["raw f0 std", "raw f1 std"], std_b8, color="gray") # Plot raw feature spreads.
plt.title("Basic 8: raw feature spreads") # Title the spread chart.
plt.show() # Display the chart.

▶ What you'll see: feature 1's raw standard deviation dwarfs feature 0's.

👀 Takeaway: kNN distances should usually be computed after fitting scaling constants on the training set.

### Basic 9 — Compute empirical risk from losses

**Goal.** Average per-example losses, because model selection starts from empirical risk on a dataset. We build it in 2 steps.

In [ ]:
losses_b9 = np.array([0.202, 0.096, 0.437]) # Use the verified toy losses from the lesson content.

print("losses:", losses_b9) # Inspect the three per-example costs.
print("sum:", round(float(np.sum(losses_b9)), 3)) # Inspect the numerator of the average.

▶ What you'll see: the losses sum to 0.735.

In [ ]:
risk_b9 = float(np.mean(losses_b9)) # Average losses into empirical risk.

print("empirical risk:", round(risk_b9, 3)) # Inspect the score before complexity costs.

assert round(risk_b9, 3) == 0.245 # Verify the content-block number.

▶ What you'll see: the average empirical loss is 0.245.

In [ ]:
plt.figure(figsize=(4.5, 3)) # Create a compact empirical-risk plot.
plt.bar(np.arange(len(losses_b9)), losses_b9, color="slateblue") # Show each per-example loss.
plt.axhline(risk_b9, color="crimson", linestyle="--", label=f"mean={risk_b9:.3f}") # Mark the empirical risk.
plt.title("Basic 9: empirical risk averages losses") # Title the loss plot.
plt.xlabel("example index"); plt.ylabel("loss"); plt.legend(); plt.show() # Display losses and their mean.

▶ What you'll see: the dashed line is the average height of the three loss bars.

👀 Takeaway: empirical risk is the mean of per-example losses, not a single hand-picked outcome.

### Basic 10 — Add a selection cost

**Goal.** Add the method's cost to raw empirical risk, because the lesson compares complete decision scores rather than training losses alone. We build it in 2 steps.

In [ ]:
risk_b10 = 0.245 # Use the rounded empirical risk from Basic 9.
cost_b10 = 0.070 # Use the lesson's complexity or operational cost.
score_b10 = risk_b10 + cost_b10 # Compute the score that drives selection.

print("risk:", risk_b10, "cost:", cost_b10) # Inspect both pieces.

▶ What you'll see: the raw fit and the extra cost are separate quantities.

In [ ]:
flex_b10 = 0.351 # A tempting alternative decision score.
gap_b10 = flex_b10 - score_b10 # Compute how much worse the alternative is.

print("baseline score:", round(score_b10, 3), "alternative:", flex_b10, "gap:", round(gap_b10, 3)) # Inspect the comparison.

assert round(score_b10, 3) == 0.315 and round(gap_b10, 3) == 0.036 # Verify the content-block arithmetic.

▶ What you'll see: the baseline score is 0.315, which beats the flexible alternative by 0.036.

In [ ]:
plt.figure(figsize=(4.5, 3)) # Create a compact decision-score comparison.
plt.bar(["baseline", "alternative"], [score_b10, flex_b10], color=["seagreen", "indianred"]) # Compare full selection scores.
plt.ylabel("decision score (lower is better)") # Label the score axis.
plt.title("Basic 10: cost-adjusted model choice") # Title the score plot.
plt.show() # Display the model-selection comparison.

▶ What you'll see: the baseline bar is lower after risk and cost are combined.

👀 Takeaway: kNN settings should be compared on full validation-style scores, not raw training fit alone.

## 🟡 Easy

### Easy 1 — Implement kNN classification end to end

**Goal.** Write the full classification rule in NumPy, because the formula is only useful when distance, sorting, and voting work together. We build it in 3 steps.

In [ ]:
X_e1 = np.array([[0.0, 0.2], [0.4, 0.1], [0.2, 0.7], [2.0, 2.2], [2.4, 1.9], [1.7, 2.5]]) # Store training examples.
y_e1 = np.array([0, 0, 0, 1, 1, 1]) # Store class labels.
q_e1 = np.array([1.4, 1.6]) # Choose the query.

print("query:", q_e1) # Inspect the prediction target.

▶ What you'll see: the query is between clusters but closer to class 1.

In [ ]:
d_e1 = np.sqrt(np.sum((X_e1 - q_e1) ** 2, axis=1)) # Compute distances to all training rows.
nn_e1 = np.argsort(d_e1)[:3] # Keep k=3 nearest indices.

print("nearest indices:", nn_e1) # Inspect N_k(x).
print("nearest labels:", y_e1[nn_e1]) # Inspect local votes.

In [ ]:
counts_e1 = np.bincount(y_e1[nn_e1], minlength=2) # Count class votes.
pred_e1 = int(np.argmax(counts_e1)) # Pick the majority class.

print("prediction:", pred_e1, "counts:", counts_e1) # Inspect the final classification.

assert pred_e1 == 1 # Verify the prediction.
plt.figure(figsize=(4, 3)) # Create a compact neighbor plot.
plt.scatter(X_e1[:, 0], X_e1[:, 1], c=y_e1, cmap="coolwarm", s=80) # Draw training points.
plt.scatter([q_e1[0]], [q_e1[1]], marker="*", s=150, color="black") # Draw the query.
plt.title("Easy 1: k=3 classification") # Title the plot.
plt.show() # Display the classification geometry.

▶ What you'll see: the three nearest neighbors belong to class 1, so the query is classified as class 1.

👀 Takeaway: kNN classification is distance calculation plus nearest-neighbor majority vote.

### Easy 2 — Predict with distance-weighted regression

**Goal.** Weight closer neighbors more heavily, because equal averaging ignores that the nearest neighbor is usually more relevant than the kth neighbor. We build it in 3 steps.

In [ ]:
X_e2 = np.array([[0.0], [1.0], [2.0], [3.0], [4.0]]) # One-dimensional training inputs.
y_e2 = np.array([1.0, 1.4, 1.9, 4.2, 3.9]) # Numeric targets.
q_e2 = np.array([2.6]) # Query location.
d_e2 = np.abs(X_e2[:, 0] - q_e2[0]) # Distances in one dimension.

print("distances:", np.round(d_e2, 2)) # Inspect local geometry.

▶ What you'll see: the point at x=3 is closest to the query.

In [ ]:
nn_e2 = np.argsort(d_e2)[:3] # Select three nearest neighbors.
w_e2 = 1.0 / (d_e2[nn_e2] + 1e-9) # Convert distance to inverse-distance weights.

print("neighbors:", nn_e2, "weights:", np.round(w_e2, 3)) # Inspect weight strength.

In [ ]:
pred_e2 = float(np.sum(w_e2 * y_e2[nn_e2]) / np.sum(w_e2)) # Weighted average of neighbor targets.
equal_e2 = float(np.mean(y_e2[nn_e2])) # Equal-weight average for comparison.

print("weighted prediction:", round(pred_e2, 3), "equal average:", round(equal_e2, 3)) # Inspect both predictions.

assert pred_e2 > equal_e2 # Verify the closest high target pulls the weighted prediction upward.
plt.figure(figsize=(4, 3)) # Create a weight chart.
plt.bar([str(i) for i in nn_e2], w_e2, color="teal") # Show inverse-distance weights.
plt.title("Easy 2: closer neighbors get larger weights") # Title the plot.
plt.xlabel("neighbor index"); plt.ylabel("weight"); plt.show() # Display the weights.

▶ What you'll see: the nearest neighbor has the largest weight and pulls the prediction toward its target.

👀 Takeaway: distance weighting keeps the prediction local while still using several neighbors.

### Easy 3 — Draw a 2-D decision region

**Goal.** Classify a grid of points, because visual decision regions reveal how kNN creates flexible, local boundaries. We build it in 3 steps.

In [ ]:
X_e3 = np.array([[0.0, 0.2], [0.4, 0.1], [0.2, 0.7], [2.0, 2.2], [2.4, 1.9], [1.7, 2.5]]) # Training points.
y_e3 = np.array([0, 0, 0, 1, 1, 1]) # Training labels.
gx_e3, gy_e3 = np.meshgrid(np.linspace(-0.5, 3.0, 45), np.linspace(-0.4, 3.0, 45)) # Make a grid of candidate queries.
grid_e3 = np.c_[gx_e3.ravel(), gy_e3.ravel()] # Flatten grid into query rows.

print("grid queries:", grid_e3.shape[0]) # Inspect how many points will be classified.

▶ What you'll see: the grid contains many tiny query locations.

In [ ]:
preds_e3 = [] # Store one kNN class per grid point.
for q_e3 in grid_e3: # Classify each grid query.
    d_e3 = np.sqrt(np.sum((X_e3 - q_e3) ** 2, axis=1)) # Distance from grid point to training data.
    nn_e3 = np.argsort(d_e3)[:3] # k=3 nearest neighbors.
    preds_e3.append(int(np.argmax(np.bincount(y_e3[nn_e3], minlength=2)))) # Majority vote.
Z_e3 = np.array(preds_e3).reshape(gx_e3.shape) # Reshape predictions for plotting.

print("class counts on grid:", np.bincount(np.array(preds_e3), minlength=2)) # Inspect region sizes.

In [ ]:
plt.figure(figsize=(4.5, 3.6)) # Create a compact decision-region plot.
plt.contourf(gx_e3, gy_e3, Z_e3, levels=[-0.5, 0.5, 1.5], alpha=0.25, colors=["steelblue", "darkorange"]) # Paint predicted regions.
plt.scatter(X_e3[:, 0], X_e3[:, 1], c=y_e3, cmap="coolwarm", edgecolor="black", s=80) # Overlay training points.
plt.title("Easy 3: kNN decision region") # Title the plot.
plt.xlabel("feature 0"); plt.ylabel("feature 1"); plt.show() # Display the regions.

▶ What you'll see: the boundary bends around the stored examples rather than following a single straight line.

👀 Takeaway: kNN decision boundaries are local and can be highly flexible.

### Easy 4 — Tune k on validation data

**Goal.** Compare several `k` values on held-out points, because the best neighborhood size is a validation choice rather than a training guess. We build it in 3 steps.

In [ ]:
X_train_e4 = np.array([[0.0, 0.2], [0.4, 0.1], [0.2, 0.7], [2.0, 2.2], [2.4, 1.9], [1.7, 2.5]]) # Training examples.
y_train_e4 = np.array([0, 0, 0, 1, 1, 1]) # Training labels.
X_val_e4 = np.array([[0.3, 0.4], [2.1, 2.0], [1.2, 1.3], [0.1, 0.9]]) # Validation queries.
y_val_e4 = np.array([0, 1, 1, 0]) # Validation labels.
ks_e4 = np.array([1, 3, 5]) # Candidate neighborhood sizes.

print("validation size:", len(y_val_e4)) # Inspect held-out count.

▶ What you'll see: validation examples are separate from stored training examples.

In [ ]:
errs_e4 = [] # Store validation error for each k.
for k_e4 in ks_e4: # Evaluate each neighborhood size.
    pred_list_e4 = [] # Store predictions for this k.
    for q_e4 in X_val_e4: # Classify each validation point.
        d_e4 = np.sqrt(np.sum((X_train_e4 - q_e4) ** 2, axis=1)) # Distances to training examples.
        nn_e4 = np.argsort(d_e4)[:k_e4] # k nearest training points.
        pred_list_e4.append(int(np.argmax(np.bincount(y_train_e4[nn_e4], minlength=2)))) # Majority vote.
    errs_e4.append(float(np.mean(np.array(pred_list_e4) != y_val_e4))) # Misclassification rate.

print("validation errors:", np.round(errs_e4, 3)) # Inspect performance by k.

In [ ]:
best_k_e4 = int(ks_e4[int(np.argmin(errs_e4))]) # Select the lowest-error k.

print("best k:", best_k_e4) # Inspect selected k.

plt.figure(figsize=(4, 3)) # Create a k-sweep plot.
plt.plot(ks_e4, errs_e4, marker="o", color="purple") # Plot error versus k.
plt.axvline(best_k_e4, color="red", linestyle="--") # Mark selected k.
plt.title("Easy 4: validation error by k") # Title the plot.
plt.xlabel("k"); plt.ylabel("validation error"); plt.xticks(ks_e4); plt.show() # Display the sweep.

▶ What you'll see: one neighborhood size has the lowest held-out error.

👀 Takeaway: choose `k` with validation data because training fit alone favors overly local rules.

### Easy 5 — Compare raw and scaled kNN

**Goal.** Run the same query before and after standardization, because scale can change the neighbor ranking and therefore the prediction. We build it in 3 steps.

In [ ]:
X_e5 = np.array([[0.0, 100.0], [1.0, 120.0], [2.0, 900.0], [3.0, 920.0]]) # Mixed-scale features.
y_e5 = np.array([0, 0, 1, 1]) # Labels for two regions.
q_e5 = np.array([2.2, 130.0]) # Query point.
raw_d_e5 = np.sqrt(np.sum((X_e5 - q_e5) ** 2, axis=1)) # Raw Euclidean distances.
raw_pred_e5 = int(y_e5[np.argmin(raw_d_e5)]) # 1-NN prediction before scaling.

print("raw distances:", np.round(raw_d_e5, 2), "raw prediction:", raw_pred_e5) # Inspect raw result.

▶ What you'll see: raw distances are dominated by the second coordinate.

In [ ]:
mu_e5 = X_e5.mean(axis=0) # Training-set feature means.
sigma_e5 = X_e5.std(axis=0) # Training-set feature standard deviations.
Z_e5 = (X_e5 - mu_e5) / sigma_e5 # Scale training data.
zq_e5 = (q_e5 - mu_e5) / sigma_e5 # Scale query with the same constants.
scaled_d_e5 = np.sqrt(np.sum((Z_e5 - zq_e5) ** 2, axis=1)) # Scaled distances.
scaled_pred_e5 = int(y_e5[np.argmin(scaled_d_e5)]) # 1-NN prediction after scaling.

print("scaled distances:", np.round(scaled_d_e5, 3), "scaled prediction:", scaled_pred_e5) # Inspect scaled result.

In [ ]:
plt.figure(figsize=(4.5, 3)) # Create a raw-vs-scaled distance plot.
plt.plot(raw_d_e5 / raw_d_e5.max(), marker="o", label="raw normalized") # Plot normalized raw distances.
plt.plot(scaled_d_e5 / scaled_d_e5.max(), marker="s", label="scaled normalized") # Plot normalized scaled distances.
plt.title("Easy 5: scaling can rerank neighbors") # Title the plot.
plt.xlabel("training index"); plt.ylabel("relative distance"); plt.legend(); plt.show() # Display the comparison.

▶ What you'll see: the relative neighbor distances change after standardization.

👀 Takeaway: scaling is part of the kNN model because it changes the geometry used for prediction.

## 🔴 Advanced

### Advanced 1 — Leave-one-out error for k selection

**Goal.** Estimate generalization using leave-one-out validation, because each training point can be tested by temporarily removing itself from memory. We build it in 4 steps.

In [ ]:
X_a1 = np.array([[0.0, 0.2], [0.4, 0.1], [0.2, 0.7], [2.0, 2.2], [2.4, 1.9], [1.7, 2.5], [1.1, 1.2]]) # Add one ambiguous middle point.
y_a1 = np.array([0, 0, 0, 1, 1, 1, 0]) # Labels include one potentially noisy point.
ks_a1 = np.array([1, 3, 5]) # Candidate k values.

print("training points:", len(y_a1)) # Inspect leave-one-out sample size.

▶ What you'll see: every stored point will become a test point once.

In [ ]:
loo_errors_a1 = [] # Store leave-one-out error rates.
for k_a1 in ks_a1: # Evaluate each k.
    wrong_a1 = 0 # Count mistakes.
    for i_a1 in range(len(y_a1)): # Hold out each point once.
        train_mask_a1 = np.arange(len(y_a1)) != i_a1 # Exclude the held-out point.
        d_a1 = np.sqrt(np.sum((X_a1[train_mask_a1] - X_a1[i_a1]) ** 2, axis=1)) # Distances to remaining points.
        nn_local_a1 = np.argsort(d_a1)[:k_a1] # Nearest remaining neighbors.
        pred_a1 = int(np.argmax(np.bincount(y_a1[train_mask_a1][nn_local_a1], minlength=2))) # Vote.
        wrong_a1 += int(pred_a1 != y_a1[i_a1]) # Accumulate mistakes.
    loo_errors_a1.append(wrong_a1 / len(y_a1)) # Store error rate.

print("LOO errors:", np.round(loo_errors_a1, 3)) # Inspect validation estimates.

In [ ]:
best_k_a1 = int(ks_a1[int(np.argmin(loo_errors_a1))]) # Select the lowest leave-one-out error.

print("best leave-one-out k:", best_k_a1) # Inspect selected k.

assert best_k_a1 in ks_a1 # Sanity-check that selection came from the grid.

In [ ]:
plt.figure(figsize=(4.5, 3)) # Create a leave-one-out curve.
plt.plot(ks_a1, loo_errors_a1, marker="o", color="navy") # Plot error by k.
plt.axvline(best_k_a1, color="crimson", linestyle="--") # Mark selected k.
plt.title("Advanced 1: leave-one-out k selection") # Title the plot.
plt.xlabel("k"); plt.ylabel("LOO error"); plt.xticks(ks_a1); plt.show() # Display the curve.

▶ What you'll see: leave-one-out error reveals which neighborhood size best predicts points not allowed to vote for themselves.

👀 Takeaway: validation must remove the example being predicted; otherwise kNN can memorize with distance zero.

### Advanced 2 — Bias-variance through repeated noisy samples

**Goal.** Compare `k=1` and a larger `k` across repeated noisy training sets, because neighborhood size controls prediction variability. We build it in 4 steps.

In [ ]:
rng_a2 = np.random.default_rng(0) # Use a local generator for reproducible repeated samples.
query_a2 = np.array([0.52]) # Fixed query for regression.
reps_a2 = 80 # Number of repeated training sets.
ks_a2 = [1, 9] # Compare very local and smoother neighborhoods.

print("query:", query_a2[0], "repetitions:", reps_a2) # Inspect simulation settings.

▶ What you'll see: the same query will be predicted from many noisy samples.

In [ ]:
preds_by_k_a2 = {1: [], 9: []} # Store predictions for each k.
for rep_a2 in range(reps_a2): # Repeat the data-generating process.
    X_train_a2 = rng_a2.random((35, 1)) # Random 1-D inputs.
    y_train_a2 = np.sin(2 * np.pi * X_train_a2[:, 0]) + 0.25 * rng_a2.normal(size=35) # Noisy targets.
    d_a2 = np.abs(X_train_a2[:, 0] - query_a2[0]) # Distances to fixed query.
    order_a2 = np.argsort(d_a2) # Neighbor ranking.
    for k_a2 in ks_a2: # Predict with each k.
        preds_by_k_a2[k_a2].append(float(np.mean(y_train_a2[order_a2[:k_a2]]))) # Local average.

print("mean predictions:", {k: round(float(np.mean(v)), 3) for k, v in preds_by_k_a2.items()}) # Inspect average behavior.

In [ ]:
stds_a2 = np.array([np.std(preds_by_k_a2[k]) for k in ks_a2]) # Variability across repeated samples.

print("prediction std by k:", dict(zip(ks_a2, np.round(stds_a2, 3)))) # Inspect variance proxy.

assert stds_a2[0] > stds_a2[1] # Verify k=1 is more variable in this simulation.

In [ ]:
plt.figure(figsize=(4.5, 3)) # Create a variability chart.
plt.hist(preds_by_k_a2[1], bins=15, alpha=0.6, label="k=1") # Histogram for local predictions.
plt.hist(preds_by_k_a2[9], bins=15, alpha=0.6, label="k=9") # Histogram for smoother predictions.
plt.title("Advanced 2: larger k reduces variance") # Title the plot.
plt.xlabel("prediction at x=0.52"); plt.ylabel("count"); plt.legend(); plt.show() # Display distributions.

▶ What you'll see: `k=1` predictions are spread wider; `k=9` is more stable but more averaged.

👀 Takeaway: increasing `k` trades local flexibility for lower prediction variance.

### Advanced 3 — High-dimensional distance concentration

**Goal.** Measure nearest and farthest distances as dimension grows, because kNN becomes harder when all points look similarly far away. We build it in 3 steps.

In [ ]:
rng_a3 = np.random.default_rng(3) # Reproducible random points.
dims_a3 = np.array([2, 10, 50, 100]) # Feature dimensions to compare.
ratios_a3 = [] # Store nearest/farthest distance ratios.

print("dimensions:", dims_a3) # Inspect dimension grid.

▶ What you'll see: the experiment moves from 2 dimensions to 100 dimensions.

In [ ]:
for d_a3 in dims_a3: # Loop over dimensions.
    X_a3 = rng_a3.normal(size=(500, d_a3)) # Random cloud of stored points.
    q_a3 = rng_a3.normal(size=d_a3) # Random query point.
    dist_a3 = np.sqrt(np.sum((X_a3 - q_a3) ** 2, axis=1)) # Distances to all points.
    ratios_a3.append(float(np.min(dist_a3) / np.max(dist_a3))) # Ratio near 1 means distances concentrate.

print("nearest/farthest ratios:", np.round(ratios_a3, 3)) # Inspect concentration.

In [ ]:
plt.figure(figsize=(4.5, 3)) # Create a concentration plot.
plt.plot(dims_a3, ratios_a3, marker="o", color="darkorange") # Plot ratio by dimension.
plt.title("Advanced 3: distances concentrate") # Title the plot.
plt.xlabel("dimension"); plt.ylabel("nearest distance / farthest distance"); plt.ylim(0, 1); plt.show() # Display the curve.

▶ What you'll see: the nearest/farthest ratio rises with dimension, making neighbor contrast weaker.

👀 Takeaway: high-dimensional kNN often needs feature selection, dimensionality reduction, or a better metric.

### Advanced 4 — Use a custom distance metric

**Goal.** Compare Euclidean and weighted Euclidean distance, because domain knowledge can say one feature should matter more than another. We build it in 3 steps.

In [ ]:
X_a4 = np.array([[0.0, 0.0], [1.0, 3.0], [2.0, 1.0], [3.0, 3.0]]) # Four candidate neighbors.
y_a4 = np.array([0, 0, 1, 1]) # Their labels.
q_a4 = np.array([1.8, 2.6]) # Query point.
weights_a4 = np.array([4.0, 0.25]) # Make feature 0 more important and feature 1 less important.

print("metric weights:", weights_a4) # Inspect custom metric weights.

▶ What you'll see: the custom metric intentionally changes feature importance.

In [ ]:
euclid_a4 = np.sqrt(np.sum((X_a4 - q_a4) ** 2, axis=1)) # Ordinary Euclidean distances.
weighted_a4 = np.sqrt(np.sum(weights_a4 * (X_a4 - q_a4) ** 2, axis=1)) # Weighted Euclidean distances.

print("euclidean:", np.round(euclid_a4, 3)) # Inspect ordinary distances.
print("weighted:", np.round(weighted_a4, 3)) # Inspect metric-adjusted distances.

In [ ]:
pred_euclid_a4 = int(y_a4[np.argmin(euclid_a4)]) # 1-NN under ordinary distance.
pred_weighted_a4 = int(y_a4[np.argmin(weighted_a4)]) # 1-NN under weighted distance.

print("pred euclidean:", pred_euclid_a4, "pred weighted:", pred_weighted_a4) # Inspect metric effect.

plt.figure(figsize=(4.5, 3)) # Create a distance comparison chart.
plt.plot(euclid_a4, marker="o", label="Euclidean") # Plot ordinary distances.
plt.plot(weighted_a4, marker="s", label="weighted") # Plot custom distances.
plt.title("Advanced 4: metric choice changes neighbors") # Title the plot.
plt.xlabel("training index"); plt.ylabel("distance"); plt.legend(); plt.show() # Display comparison.

▶ What you'll see: changing the metric changes the distance ranking and can change the 1-NN label.

👀 Takeaway: the distance metric encodes assumptions about what "similar" should mean.

### Advanced 5 — Visualize underfitting and overfitting with k

**Goal.** Compare decision regions for small and large `k`, because kNN capacity is visible in the smoothness of the boundary. We build it in 4 steps.

In [ ]:
rng_a5 = np.random.default_rng(5) # Reproducible synthetic classification data.
angles_a5 = rng_a5.uniform(0, 2 * np.pi, 60) # Random angles.
radii_a5 = rng_a5.uniform(0.2, 1.4, 60) # Random radii.
X_a5 = np.c_[radii_a5 * np.cos(angles_a5), radii_a5 * np.sin(angles_a5)] # Points in a disk.
y_a5 = (radii_a5 > 0.8).astype(int) # Outer ring is class 1, inner disk is class 0.
flip_a5 = rng_a5.choice(60, 5, replace=False) # Add a few noisy labels.
y_a5[flip_a5] = 1 - y_a5[flip_a5] # Flip selected labels.

print("class counts:", np.bincount(y_a5, minlength=2)) # Inspect noisy class balance.

▶ What you'll see: there are inner/outer classes with a few deliberate label mistakes.

In [ ]:
grid_x_a5, grid_y_a5 = np.meshgrid(np.linspace(-1.6, 1.6, 70), np.linspace(-1.6, 1.6, 70)) # Prediction grid.
grid_a5 = np.c_[grid_x_a5.ravel(), grid_y_a5.ravel()] # Flatten grid into query rows.
ks_show_a5 = [1, 15] # Compare highly flexible and smoother neighborhoods.
regions_a5 = [] # Store region predictions.

print("grid size:", grid_a5.shape[0]) # Inspect number of query points.

In [ ]:
for k_a5 in ks_show_a5: # Build one decision map per k.
    preds_a5 = [] # Store grid predictions for this k.
    for q_a5 in grid_a5: # Classify every grid point.
        d_a5 = np.sqrt(np.sum((X_a5 - q_a5) ** 2, axis=1)) # Distances to training points.
        nn_a5 = np.argsort(d_a5)[:k_a5] # k nearest neighbors.
        preds_a5.append(int(np.argmax(np.bincount(y_a5[nn_a5], minlength=2)))) # Majority vote.
    regions_a5.append(np.array(preds_a5).reshape(grid_x_a5.shape)) # Reshape for contour plotting.

print("region means:", [round(float(r.mean()), 3) for r in regions_a5]) # Inspect fraction predicted class 1.

In [ ]:
fig_a5, ax_a5 = plt.subplots(1, 2, figsize=(8, 3.4)) # Create side-by-side decision maps.
for j_a5, k_a5 in enumerate(ks_show_a5): # Plot each k.
    ax_a5[j_a5].contourf(grid_x_a5, grid_y_a5, regions_a5[j_a5], levels=[-0.5, 0.5, 1.5], alpha=0.25, colors=["steelblue", "darkorange"]) # Paint regions.
    ax_a5[j_a5].scatter(X_a5[:, 0], X_a5[:, 1], c=y_a5, cmap="coolwarm", s=25, edgecolor="k", linewidth=0.2) # Overlay data.
    ax_a5[j_a5].set_title(f"k={k_a5}") # Label each panel.
plt.suptitle("Advanced 5: small k vs large k boundaries") # Overall title.
plt.show() # Display the comparison.

▶ What you'll see: `k=1` makes jagged islands around noisy labels, while `k=15` makes a smoother ring-like boundary.

👀 Takeaway: small `k` can overfit local noise; large `k` can underfit by smoothing away real local structure.